# J3 après-midi — Enrichir des données avec une API
**13h30–17h00 · Programmation en Python · Mastère 1 Data & IA**

Comment transformer une donnée externe récupérée sur le web en information exploitable dans notre dataset ?

**Objectifs :** Consommer une API REST avec requests : comprendre GET/POST, format JSON, authentification par token (récupérer des données météo, financières, ou d'une API publique). Stocker le résultat dans un DataFrame et l'intégrer au dataset fil rouge

## 0. Démarrage
Ouvrez ce notebook dans Colab. Dans le volet **Fichiers → Importer**, déposez :

- le **`sales_analysis.csv` déjà utilisé en J2 après-midi**, sans le modifier ;
- `api_fallback.json`, fourni avec ce notebook.

Exécutez les cellules dans l’ordre avec **Maj + Entrée**. Chaque question possède sa cellule `# TODO`.
Après une réinitialisation de Colab, réimportez les deux ressources. Aucun environnement local ni compte API nécessaire.

Les cellules de secours sont commentées : elles ne remplacent les appels réels qu’en cas de panne.

In [1]:
import requests
import pandas as pd

## 1. Premier GET · 13h30–13h45
**Démonstration et échanges : 15 min.** JSONPlaceholder fournit des données fictives pour apprendre à interroger un service.

Le notebook envoie une **requête HTTP** au serveur. Le serveur renvoie une **réponse**.
Ici, nous demandons les commentaires associés à la publication numéro 1.

[Documentation JSONPlaceholder](https://jsonplaceholder.typicode.com/guide/)

In [2]:
#Site qui permet de requeter des API
url = "https://jsonplaceholder.typicode.com/comments"

params = {
    "postId": 1
}

In [9]:
import requests
import json

url = "https://jsonplaceholder.typicode.com/comments"

params = {
    "postId": 1
}

response = requests.get(url, params=params)

data = response.json()

print(json.dumps(data, indent=4, ensure_ascii=False))

[
    {
        "postId": 1,
        "id": 1,
        "name": "id labore ex et quam laborum",
        "email": "Eliseo@gardner.biz",
        "body": "laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium"
    },
    {
        "postId": 1,
        "id": 2,
        "name": "quo vero reiciendis velit similique earum",
        "email": "Jayne_Kuhic@sydney.com",
        "body": "est natus enim nihil est dolore omnis voluptatem numquam\net omnis occaecati quod ullam at\nvoluptatem error expedita pariatur\nnihil sint nostrum voluptatem reiciendis et"
    },
    {
        "postId": 1,
        "id": 3,
        "name": "odio adipisci rerum aut animi",
        "email": "Nikita@garfield.biz",
        "body": "quia molestiae reprehenderit quasi aspernatur\naut expedita occaecati aliquam eveniet laudantium\nomnis quibusdam delectus saepe quia accusamus maiores nam est\ncum et ducimus et vero voluptates 

In [10]:
response = requests.get(
    url,
    params=params,
    timeout=20
)

**Observer la réponse.** `200` signifie que la requête a réussi. `timeout=20` limite l’attente réseau ; il ne modifie pas les données demandées.

In [11]:
response.status_code

200

**Décoder le JSON.** La réponse HTTP et les données qu’elle transporte sont deux objets différents.

In [5]:
data = response.json()

In [6]:
type(data)

list

In [7]:
data[0]

{'postId': 1,
 'id': 1,
 'name': 'id labore ex et quam laborum',
 'email': 'Eliseo@gardner.biz',
 'body': 'laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium'}

**À constater ensemble :** nous avons une liste de dictionnaires. Les clés et valeurs se manipulent avec le Python déjà connu.
`.json()` décode le contenu ; cette méthode ne vérifie pas à elle seule que la requête a réussi.

In [8]:
data[0]["body"]

'laudantium enim quasi est quidem magnam voluptate ipsam eos\ntempora quo necessitatibus\ndolor quam autem quasi\nreiciendis et nam sapiente accusantium'

## 2. Frankfurter : GET → JSON → DataFrame · 13h45–14h10
**Préparation : 5 min · exercice : 15 min · correction : 5 min.**

Les ventes sont en livres sterling. Nous voulons récupérer un taux **GBP → EUR** représentatif de chacun de leurs quatre mois.
Commençons par lire la période réelle des ventes, sans la deviner.

In [12]:
sales = pd.read_csv("sales_analysis1.csv")

sales["invoice_date"] = pd.to_datetime(sales["invoice_date"])

sales.head(3)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country,market
0,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00,4.50,12346.0,United Kingdom,UK
1,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00,4.50,12346.0,United Kingdom,UK
2,493413,21723,ALPHABET HEARTS STICKER SHEET,1,2010-01-04 09:54:00,0.85,NaN,United Kingdom,UK


In [13]:
date_min = sales["invoice_date"].min()
date_max = sales["invoice_date"].max()

start_date = date_min.strftime("%Y-%m-%d")
end_date = date_max.strftime("%Y-%m-%d")

print("Première vente :", date_min)
print("Dernière vente :", date_max)

Première vente : 2010-01-04 09:24:00
Dernière vente : 2010-04-30 17:33:00


In [14]:
sales_months = sales["invoice_date"].dt.to_period("M")
expected_months = sorted(sales_months.unique())

print("Mois des ventes :", expected_months)
print("Nombre de mois :", len(expected_months))

Mois des ventes : [Period('2010-01', 'M'), Period('2010-02', 'M'), Period('2010-03', 'M'), Period('2010-04', 'M')]
Nombre de mois : 4


### Exercice A — Obtenir les taux utiles - 15 min
**Endpoint :** `https://api.frankfurter.dev/v2/rates`

| Paramètre | Rôle dans notre demande |
|---|---|
| `from` | Début de période au format `YYYY-MM-DD` : variable `start_date` |
| `to` | Fin de période au même format : variable `end_date` |
| `base` | Devise de départ : GBP |
| `quotes` | Devise d’arrivée : EUR |
| `group` | Un point représentatif par mois : valeur `month` |

Pas de clé à fournir. L’API renvoie un JSON à inspecter avant de choisir comment le transformer.
Le regroupement mensuel est fourni par le service ; nous l’utilisons comme indicateur simplifié, pas comme une conversion comptable exacte de chaque vente.

[Documentation Frankfurter](https://frankfurter.dev/) · [Contrat de l’API](https://api.frankfurter.dev/v2/openapi.json)

In [20]:
rates_url = "https://api.frankfurter.dev/v2/rates"

In [15]:
start_date, end_date

('2010-01-04', '2010-04-30')

**A1 — Paramètres.** Construisez le dictionnaire `params` pour demander les taux GBP → EUR sur la période des ventes, avec un point par mois.

In [18]:
# TODO A1 Construisons le dictionnaire `params` pour demander les taux GBP → EUR 
# sur la période des ventes, avec un point par mois.
params= {
    "from": start_date,
    "to": end_date,
    "base": "GBP",
    "quotes": "EUR",
    "group": "month"
}

params


{'from': '2010-01-04',
 'to': '2010-04-30',
 'base': 'GBP',
 'quotes': 'EUR',
 'group': 'month'}

**A2 — Requête.** Effectuez le GET vers `rates_url` avec vos paramètres. Stockez la réponse dans `rates_response` et limitez l’attente à 20 secondes.

In [21]:
# TODO A2
rates_response = requests.get(
    rates_url,
    params,
    timeout=20
)


**A3 — Statut.** Affichez le status code. Ne poursuivez avec la réponse que si le statut est 200 ; sinon utilisez le secours placé après A4.

In [24]:
# TODO A3
rates_response.status_code

200

**A4 — Inspection.** Décodez le JSON dans `rates_data`. Affichez son type, sa taille et son premier élément. Identifiez oralement ce que décrit un élément.

In [25]:
# TODO A4
rates_data = rates_response.json()   
rates_data

[{'date': '2010-01-01', 'base': 'GBP', 'quote': 'EUR', 'rate': 1.1318},
 {'date': '2010-02-01', 'base': 'GBP', 'quote': 'EUR', 'rate': 1.1414},
 {'date': '2010-03-01', 'base': 'GBP', 'quote': 'EUR', 'rate': 1.1084},
 {'date': '2010-04-01', 'base': 'GBP', 'quote': 'EUR', 'rate': 1.1411}]

**A5 — Table.** À partir de la structure observée, construisez le DataFrame `rates`. Affichez-le et repérez les champs décrivant la date, les devises et le taux.

In [26]:
# TODO A5
sales

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country,market
0,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00,4.50,12346.0,United Kingdom,UK
1,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00,4.50,12346.0,United Kingdom,UK
2,493413,21723,ALPHABET HEARTS STICKER SHEET,1,2010-01-04 09:54:00,0.85,NaN,United Kingdom,UK
3,493413,21724,PANDA AND BUNNIES STICKER SHEET,1,2010-01-04 09:54:00,0.85,NaN,United Kingdom,UK
4,493413,84578,ELEPHANT TOY WITH BLUE T-SHIRT,1,2010-01-04 09:54:00,3.75,NaN,United Kingdom,UK
...,...,...,...,...,...,...,...,...,...
130963,506673,85049C,ROMANTIC PINKS RIBBONS,1,2010-04-30 17:33:00,1.25,17602.0,United Kingdom,UK
130964,506673,85174,S/4 CACTI CANDLES,2,2010-04-30 17:33:00,4.95,17602.0,United Kingdom,UK
130965,506673,85231B,CINAMMON SET OF 9 T-LIGHTS,3,2010-04-30 17:33:00,0.85,17602.0,United Kingdom,UK
130966,506673,85231E,STRAWBERRY SCENTED SET/9 T-LIGHTS,3,2010-04-30 17:33:00,0.85,17602.0,United Kingdom,UK


**A6 — Colonnes utiles.** Vérifiez que toutes les lignes décrivent GBP → EUR. Gardez uniquement `date` et `rate`, puis renommez le taux en `gbp_to_eur`. Affichez `rates`.

In [37]:
# TODO A6
rates = rates[["date", "rate"]]

rates = rates[["date", "rate"]].rename(
    columns={"rate": "gbp_to_eur"}
)

rates

,date,gbp_to_eur
0,2010-01-01,1.1318
1,2010-02-01,1.1414
2,2010-03-01,1.1084
3,2010-04-01,1.1411


## 3. Intégrer les taux aux ventes - 20 min
**Démarrage : 5 min · exercice : 20 min · correction : 8 min.**

Nous voulons **une ligne par semaine**, avec le montant GBP, le taux pour le mois en cours et le montant EUR.
Le calcul des ventes est fourni pour concentrer le travail sur la jointure.

In [33]:
sales["line_total_gbp"] = (
    sales["quantity"] * sales["unit_price_gbp"]
)

sales["week"] = sales["invoice_date"].dt.to_period("W")


In [30]:
weekly_sales = sales.groupby("week", as_index=False).agg(
    total_sales_gbp=("line_total_gbp", "sum")
)

weekly_sales

,week,total_sales_gbp
0,2010-01-04/2010-01-10,168228.580
1,2010-01-11/2010-01-17,163595.570
2,2010-01-18/2010-01-24,153725.871
3,2010-01-25/2010-01-31,165605.091
4,2010-02-01/2010-02-07,124084.092
5,2010-02-08/2010-02-14,90814.120
6,2010-02-15/2010-02-21,174496.562
7,2010-02-22/2010-02-28,162109.952
8,2010-03-01/2010-03-07,165865.590
9,2010-03-08/2010-03-14,141839.540


### Exercice B — Faire correspondre les mois
Une date écrite en texte et une période mensuelle peuvent sembler similaires à l’écran, tout en étant différentes pour Pandas.
Nous choisissons une convention commune : **le premier jour du mois, en datetime**.
Réactivation : `.dt.to_period("M")` représente un mois ; `.dt.to_timestamp()` le ramène à son premier jour.

In [38]:
print("Ventes :", weekly_sales["week"].dtype)
print("API :", rates["date"].dtype)

Ventes : period[W-SUN]
API : object


**B1 — Clé côté ventes.** Convertissez la colonne `week` de `weekly_sales` selon la convention choisie. Affichez la table et vérifiez le type de cette colonne.

In [40]:

# TODO B1
weekly_sales["week"] = (
    weekly_sales["week"]
    .dt.to_timestamp(how="start")
    .dt.to_period("M")
    .dt.to_timestamp(how="start")
)

weekly_sales


,week,total_sales_gbp
0,2010-01-01,168228.580
1,2010-01-01,163595.570
2,2010-01-01,153725.871
3,2010-01-01,165605.091
4,2010-02-01,124084.092
5,2010-02-01,90814.120
6,2010-02-01,174496.562
7,2010-02-01,162109.952
8,2010-03-01,165865.590
9,2010-03-01,141839.540


**B2 — Clé côté taux.** Convertissez `rates["date"]` selon la convention choisie (compatible avec celle des ventes)et affichez la table.

In [ ]:
# TODO B2


**B3.Test — Couverture.** Comparez les dates des deux tables. Vérifiez que les quatre mois sont présents et que chaque mois possède un seul taux avant toute jointure.

In [ ]:
## TEST YOU CODE ##

# clé primaire ? 4 rates différents
assert rates["date"].is_unique
assert len(rates) == 4

# vérifier qu'on a les mêmes objets dans les 2 dataframes
sales_week_set = set(weekly_sales["week"])
rates_dates_set = set(rates["date"])

assert sales_week_set == rates_dates_set
print("Quatre mois couverts, un taux par mois.")

**B4 — Jointure.** Construisez `weekly_sales_eur` en rattachant `gbp_to_eur` aux ventes par `week`. Conservez tous les mois des ventes et vérifiez une correspondance un-à-un avec `validate="many_to_one"`.

In [ ]:
# TODO B4


**Sens du taux.** Avec `base=GBP` et `quote=EUR`, le taux exprime le nombre d’euros pour **1 GBP**.
La conversion applique ce nombre à chaque livre : par exemple, un taux de 1,13 signifie que 100 GBP correspondent à 113 EUR.
Les taux observés pour nos mois sont proches de cet ordre de grandeur. Il s’agit d’un enrichissement d’indicateurs avec un taux mensuel représentatif.

**B5 — Conversion.** Créez `total_sales_eur` dans `weekly_sales_eur` à partir du montant GBP et du taux, en respectant leur sens. Affichez les quatre mois ; conservez la précision du calcul. 

N'oubliez pas de retirer les colonnes redondantes...

In [ ]:
# TODO B5


**B6.Tests — Contrôles.** Vérifiez : nombre de lignes conservé, aucun mois dupliqué ou perdu, aucun taux ni montant EUR manquant, taux positifs. Comparez aussi le total GBP avant et après la jointure.

In [ ]:
## TEST YOU CODE ##

assert len(weekly_sales_eur) == len(weekly_sales)
assert weekly_sales_eur["gbp_to_eur"].notna().all()
assert weekly_sales_eur["gbp_to_eur"].gt(0).all()
assert weekly_sales_eur["total_sales_eur"].notna().all()

before = weekly_sales["total_sales_gbp"].sum()
after = weekly_sales_eur["total_sales_gbp"].sum()
assert abs(before - after) < 0.01

print("Jointure validée : aucune semaine perdue ni multiplié.")

## 4. POST, token et erreur HTTP · 14h45–14h55
**Trois mini-démonstrations : 10 min au total.**

### Envoyer des données : POST
GET récupère une ressource ; POST envoie des données au serveur.
JSONPlaceholder simule la création : le résultat renvoyé **n’est pas enregistré durablement**.

In [ ]:
payload = {
    "title": "Synthèse des ventes",
    "body": "Analyse mensuelle disponible",
    "userId": 1
}

In [ ]:
post_response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=payload,
    timeout=20
)

In [ ]:
post_response.status_code

In [ ]:
post_response.json()

### Authentification par token
Certains services demandent un token dans les en-têtes. C’est un **secret** : ne l’écrivez pas dans un notebook partagé.
Le texte ci-dessous est fictif. Aucun compte, secret ou appel authentifié n’est nécessaire ici.

In [ ]:
headers = {
    "Authorization": "Bearer MON_TOKEN"
}

# Forme illustrative seulement, aucun appel à exécuter :
# requests.get(url_protegee, headers=headers, timeout=20)

### Repérer une erreur HTTP
Demandons une publication inexistante. Un statut 404 indique que la ressource n’a pas été trouvée.

In [ ]:
error_response = requests.get(
    "https://jsonplaceholder.typicode.com/posts/999999",
    timeout=20
)

error_response.status_code

`raise_for_status()` transforme un statut HTTP d’erreur en exception Python.
Nous interceptons uniquement l’erreur attendue pour pouvoir poursuivre l’exécution du notebook.

In [ ]:
try:
    error_response.raise_for_status()
except requests.HTTPError:
    print("HTTPError : la ressource demandée est introuvable.")

## 5. Mission — La météo des magasins · 14h55–15h15
**Travail autonome : 14 min · restitution et correction : 6 min.**

**Mission : enrichissez `stores` avec la température actuelle de chacun des trois magasins.**
Les magasins sont fictifs ; leurs coordonnées correspondent aux centres de Paris, Berlin et Madrid.

Cette fois, les paramètres dépendent des lignes d’une table et la réponse contient des objets imbriqués. L’exploration doit permettre d’identifier les données utiles.

**Endpoint :** `https://api.open-meteo.com/v1/forecast`

| Paramètre | Valeur à transmettre |
|---|---|
| `latitude` | Latitude du magasin |
| `longitude` | Longitude du magasin |
| `current` | `temperature_2m` : température actuelle à deux mètres |

Résultat attendu : une ligne par magasin, avec `store_id`, `city`, `latitude`, `longitude`, `temperature_2m` en °C.
Les données actuelles proviennent des modèles météo ; les valeurs varient au fil du temps. Aucun token nécessaire pour cet usage pédagogique.

[Documentation Open-Meteo](https://open-meteo.com/en/docs)

In [ ]:
stores = pd.DataFrame({
    "store_id": ["PAR", "BER", "MAD"],
    "city": ["Paris", "Berlin", "Madrid"],
    "latitude": [48.8566, 52.5200, 40.4168],
    "longitude": [2.3522, 13.4050, -3.7038]
})

stores

**C1 — Explorer un magasin.** Sur le premier magasin, réalisez un GET, vérifiez son statut et décodez le JSON dans `weather_data`. Inspectez la réponse pour localiser la température et son unité.

In [ ]:
# TODO C1


**C2 — Couvrir les magasins.** Construisez `weather_results` pour l’ensemble des magasins. Conservez de quoi rattacher chaque température au bon magasin. Organisez librement vos étapes ; vérifiez les réponses reçues.

In [ ]:
# TODO C2


**C3 — Structurer.** Transformez `weather_results` en DataFrame `weather`. Affichez la table.

In [ ]:
# TODO C3


**C4 — Enrichir.** Enrichissez `stores` dans `stores_weather` pour obtenir la forme attendue. Vérifiez que chaque magasin possède une température et qu’aucune ligne n’a été perdue ou multipliée.

In [ ]:
# TODO C4


## 6. Git : du fichier au commit · 16h00–16h40
**Démonstration et échanges : 20 min · compréhension : 12 min · correction : 8 min.**

## DEMO TIME ! (hors notebook)

### Exercice D — Lire et expliquer
Répondez en commentaires Python dans chaque cellule.

**D1 — Ordre.** Pour un nouveau projet avec un dépôt distant prêt à recevoir les commits, remettez dans l’ordre : `git commit`, `git init`, `git push`, `git add`.

In [ ]:
# TODO D1


**D2 — États.** Voici un exemple de `git status`. Associez chaque fichier à **untracked**, **staged** ou **modified non préparé**.

```text
Changes to be committed:
    new file:   prepare_data.py

Changes not staged for commit:
    modified:   analysis.py

Untracked files:
    notes.txt
```

In [ ]:
# TODO D2


**D3 — Choix de fichiers.** Dans ce projet, les données sont distribuées séparément et peuvent être récupérées. Parmi `analysis.py`, `README.md`, `sales_analysis.csv` et `raw_data_2gb.csv`, lesquels placeriez-vous dans `.gitignore` ? Justifiez.

In [ ]:
# TODO D3


**D4 — Nouvelle modification.** J’ai modifié `analysis.py` après mon dernier commit. Quelle étape dois-je refaire avant de pouvoir inclure cette modification dans mon prochain commit ?

In [ ]:
# TODO D4
